In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

orders = pd.read_csv('data/orders.csv')
customers = pd.read_csv('data/customers.csv')

CUTOFF  = 25
HORIZON = 6
print('Loaded', len(orders), 'orders and', len(customers), 'customers')

Loaded 16961 orders and 2000 customers


In [11]:
history = orders[orders['week'] <= CUTOFF]
recent_ids = history[history['week'] > CUTOFF - 6]['customer_id'].unique()
base = pd.DataFrame({'customer_id': recent_ids})
print('Active base:', len(base), 'customers')

Active base: 582 customers


In [12]:
h = history[history['customer_id'].isin(recent_ids)]
last8 = h[h['week'] > CUTOFF - 8]

feat = pd.DataFrame({'customer_id': recent_ids}).set_index('customer_id')
feat['weeks_since_last_order'] = CUTOFF - h.groupby('customer_id')['week'].max()
feat['orders_last_8w']  = last8.groupby('customer_id')['order_id'].count()
feat['late_last_8w']    = last8.assign(late=(~last8['delivered_on_time']).astype(int))\
                                .groupby('customer_id')['late'].sum()
feat['tenure_weeks']    = CUTOFF - customers.set_index('customer_id')['signup_week']
feat['avg_meals']       = h.groupby('customer_id')['n_meals'].mean()
feat = feat.fillna(0)
feat.head()

,weeks_since_last_order,orders_last_8w,late_last_8w,tenure_weeks,avg_meals
customer_id,,,,,
5,1,5,0,5,5.0
11,0,8,1,8,4.0
16,0,8,2,21,3.0
17,0,8,1,11,3.0
18,0,8,1,22,3.0


In [13]:
future = orders[(orders['week'] > CUTOFF) & (orders['week'] <= CUTOFF + HORIZON)]
ordered_in_future = set(future['customer_id'].unique())
feat['churned'] = (~feat.index.isin(ordered_in_future)).astype(int)
print('Churn rate in active base:', round(feat['churned'].mean(), 3))

Churn rate in active base: 0.4


In [14]:
feature_names = ['weeks_since_last_order', 'orders_last_8w',
                 'late_last_8w', 'tenure_weeks', 'avg_meals']
X = feat[feature_names]
y = feat['churned']

X_scaled = (X - X.mean()) / X.std()
X_scaled.head()

,weeks_since_last_order,orders_last_8w,late_last_8w,tenure_weeks,avg_meals
customer_id,,,,,
5,-0.004233,-0.025502,-0.828166,-0.511967,1.199959
11,-0.620065,1.177917,0.464040,-0.046397,-0.008304
16,-0.620065,1.177917,1.756245,1.971072,-1.216567
17,-0.620065,1.177917,0.464040,0.419173,-1.216567
18,-0.620065,1.177917,0.464040,2.126262,-1.216567


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y)

model = LogisticRegression()
model.fit(X_train, y_train)

pred_prob = model.predict_proba(X_test)[:, 1]
print('AUC      :', round(roc_auc_score(y_test, pred_prob), 3))
print('Accuracy :', round(accuracy_score(y_test, model.predict(X_test)), 3))

AUC      : 0.925
Accuracy : 0.926


In [16]:
coef = pd.DataFrame({
    'feature': feature_names,
    'weight': model.coef_[0]
}).sort_values('weight', ascending=False)
coef

,feature,weight
0,weeks_since_last_order,5.114781
2,late_last_8w,0.204212
4,avg_meals,0.115368
3,tenure_weeks,0.101077
1,orders_last_8w,-0.595388


In [17]:
feat['churn_risk'] = model.predict_proba(X_scaled)[:, 1]
at_risk = feat.sort_values('churn_risk', ascending=False).head(10)
at_risk[['churn_risk', 'weeks_since_last_order', 'orders_last_8w', 'late_last_8w']].round(2)

,churn_risk,weeks_since_last_order,orders_last_8w,late_last_8w
customer_id,,,,
1325,1.0,5,1,1
1296,1.0,5,3,2
839,1.0,5,3,1
1873,1.0,5,2,1
1289,1.0,5,2,2
1310,1.0,5,3,2
1523,1.0,5,1,0
1782,1.0,5,3,2
217,1.0,5,2,1
